# Infinity-2B Step-wise PFB vs Content-Orthogonal (Colab)

This notebook mounts a full `VAR-SOICT` folder from Google Drive, caches Infinity-2B Q8 GGUF, Infinity VAE, and FLAN-T5-XL encoder under `working_dir/model_dir`, then saves step-wise comparisons: **Baseline | PFB | Content-Orthogonal**.

Use a GPU runtime. An A100/L4 with sufficient VRAM is recommended.

## 1. Mount Drive and choose the working directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys

# Change only this line if the uploaded folder has another name/location.
WORKING_DIR = Path('/content/drive/MyDrive/VAR-SOICT').resolve()
MODEL_DIR = WORKING_DIR / 'model_dir'
OUTPUT_DIR = WORKING_DIR / 'outputs' / 'stepwise_pfb_content_ortho'

if not (WORKING_DIR / 'src' / 'var_soict').exists():
    raise FileNotFoundError(f'VAR-SOICT source not found under {WORKING_DIR}')
if not (WORKING_DIR / 'Infinity' / 'infinity' / 'models' / 'infinity.py').exists():
    raise FileNotFoundError(f'Vendored Infinity source not found under {WORKING_DIR / "Infinity"}')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORKING_DIR / 'src'))

print('Working directory:', WORKING_DIR)
print('Persistent model cache:', MODEL_DIR)
print('Persistent outputs:', OUTPUT_DIR)

## 2. Check GPU and install runtime dependencies

Do **not** run `pip install flash_attn` or install an old unmodified `Infinity/requirements.txt`. Current Colab Python 3.13 generally has no compatible FlashAttention wheel; this repository uses PyTorch SDPA instead.

In [ ]:
from var_soict.bootstrap import check_torch_runtime, install_dependencies

DEVICE = check_torch_runtime(require_cuda=True)
install_dependencies()

# Added explicitly because bootstrap uses hf_hub_download for the three checkpoints.
%pip install -q huggingface_hub

## 3. Configure Drive-backed model cache

In [ ]:
from var_soict.config import ExperimentConfig

config = ExperimentConfig(
    root=MODEL_DIR,
    infinity_source_dir=WORKING_DIR / 'Infinity',
    download_missing_model_files=True,
    output_run_name='stepwise_pfb_content_ortho',
    model_pn='0.25M',
    cfg=1.0,
    tau=0.1,
    top_k=600,
    top_p=0.95,
    seed=2026,
    t5_device='cuda',
)
print('Infinity source:', WORKING_DIR / 'Infinity')
print('Weights/cache only:', MODEL_DIR)

## 4. Download/cache Infinity-2B, Infinity VAE, and FLAN-T5-XL

In [ ]:
from var_soict.bootstrap import (
    import_gguf_loader_direct,
    prepare_direct_model_files,
)

INFINITY_SOURCE_DIR = WORKING_DIR / 'Infinity'
model_files = prepare_direct_model_files(
    config,
    model_dir=MODEL_DIR,
    infinity_source_dir=INFINITY_SOURCE_DIR,
)
gguf_loader = import_gguf_loader_direct(
    model_files,
    infinity_source_dir=INFINITY_SOURCE_DIR,
)

print('Infinity-2B:', model_files.infinity_gguf)
print('Infinity VAE:', model_files.vae_path)
print('FLAN-T5-XL:', model_files.t5_gguf)
print('No Infinity_runtime directory is used.')

## 5. Initialize the Infinity model

In [ ]:
from var_soict.bootstrap import build_scale_schedule, load_model_bundle

bundle = load_model_bundle(config, model_files, gguf_loader)
bundle.scale_schedule = build_scale_schedule(config.model_pn, aspect_ratio=1.0)

assert hasattr(bundle.infinity_model, 'autoregressive_infer_pfb')
assert hasattr(bundle.infinity_model, 'autoregressive_infer_content_projection')

# Independent algebraic smoke test for projected PFB (no model inference).
import torch
from var_soict.feature_hypotheses import projected_pfb_content_blend
toy_fc = torch.randn(1, 8, 1, 4, 4, device=DEVICE)
toy_fg = torch.randn_like(toy_fc)
toy_fs = torch.randn_like(toy_fc)
toy_out, toy_diag = projected_pfb_content_blend(
    toy_fg, toy_fs, toy_fc, style_rank=2, content_rank=2,
    projection_strength=1.0, preserve_mean=False, return_diagnostics=True,
)
assert toy_out.shape == toy_fg.shape
assert toy_diag[0]['orthogonality_residual'] < 1e-5, toy_diag
print('Projected-PFB smoke test:', toy_diag)
print('Number of AR resolutions:', len(bundle.scale_schedule))

## 6. Select a CSD100 content/style pair

The styled prompt deliberately keeps the content subject unchanged and adds only the style phrase. The CSD100 style image is retained as a visual reference.

In [ ]:
from IPython.display import display
from PIL import Image

from var_soict.csd100_stepwise import load_csd100_stepwise_case

# Folder names under WORKING_DIR/csd100. Change these two values freely.
CONTENT_ITEM_ID = 'fox+graffiti'
STYLE_ITEM_ID = 'pen+artwork'

case = load_csd100_stepwise_case(
    WORKING_DIR,
    content_item_id=CONTENT_ITEM_ID,
    style_item_id=STYLE_ITEM_ID,
)
print('Content prompt:', case.content_prompt)
print('Styled prompt :', case.style_prompt)
print('Style label   :', case.style_label)
display(Image.open(case.content_reference_path).convert('RGB'))
display(Image.open(case.style_reference_path).convert('RGB'))

## 7. Run the step-wise experiment

Start with a few diagnostically useful steps. Set `INJECT_STEPS = None` to run every resolution. Each selected step performs one PFB run and one content-orthogonal run.

In [ ]:
from var_soict.csd100_stepwise import run_csd100_stepwise

INJECT_STEPS = [0, 1, 2, 3, 6, 9]
SAC = False  # Rerun with True after inspecting the residual-only experiment.

case, result = run_csd100_stepwise(
    bundle,
    config,
    var_soict_root=WORKING_DIR,
    content_item_id=CONTENT_ITEM_ID,
    style_item_id=STYLE_ITEM_ID,
    output_root=OUTPUT_DIR,
    inject_steps=INJECT_STEPS,
    seed=config.seed,
    sac=SAC,
    style_rank=1,
    content_rank=1,  # fixed-rank baseline; use None with threshold below for adaptive rank
    content_variance_threshold=None,  # e.g. 0.90 with content_rank=None
    strength=1.0,
    projection_strength=1.0,
    preserve_mean=False,
)
print('Case output:', OUTPUT_DIR / case.case_name)

## 8. Display Baseline | PFB | Content-Orthogonal comparisons

In [ ]:
from var_soict.stepwise_feature_experiment import display_stepwise_comparisons

display_stepwise_comparisons(result)
for step, diagnostics in sorted(result.projection_diagnostics_by_step.items()):
    print(f'step {step}:', diagnostics)

## 9. Inspect saved artifacts

In [ ]:
case_output_dir = OUTPUT_DIR / case.case_name
for path in sorted(case_output_dir.iterdir()):
    print(path.name)

## 10. Mini benchmark: 5 content prompts × 5 CSD100 styles

The benchmark caches five clean content trajectories and five VQVAE style traces, then evaluates **Baseline**, **PFB**, and **Content Projection** at steps 2, 3, and 4. These middle/coarse-middle steps bracket the strongest useful region in the visual pilot without including the degenerate 1×1 content projector at step 0.

In [ ]:
import gc
import json
import shutil
from tqdm.auto import tqdm

from var_soict.csd100_stepwise import load_csd100_stepwise_case
from var_soict.feature_hypotheses import principal_feature_blend
from var_soict.stepwise_feature_experiment import (
    _cpu, _image, _infer_kwargs, _run_infinity, encode_style_reference,
)

BENCHMARK_CONTENT_IDS = [
    'fox+graffiti',
    'mushroom+tattoo',
    'rabbit+cubism',
    'piano+impressionism',
    'duck+blueprint',
]
BENCHMARK_STYLE_IDS = [
    'pen+artwork',
    'turtle+origami',
    'flower+pixel',
    'robot+woodcut',
    'glass+watercolor_and_ink_wash',
]
BENCHMARK_STEPS = [2, 3, 4]
BENCHMARK_DIR = OUTPUT_DIR / 'benchmark_5x5'
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

if max(BENCHMARK_STEPS) >= len(bundle.scale_schedule):
    raise ValueError(f'Benchmark steps exceed {len(bundle.scale_schedule)} AR scales')

# Resolve controlled prompts and the five independent style-reference images.
content_cases = {
    content_id: load_csd100_stepwise_case(
        WORKING_DIR, content_item_id=content_id, style_item_id=BENCHMARK_STYLE_IDS[0]
    )
    for content_id in BENCHMARK_CONTENT_IDS
}
style_cases = {
    style_id: load_csd100_stepwise_case(
        WORKING_DIR, content_item_id=BENCHMARK_CONTENT_IDS[0], style_item_id=style_id
    )
    for style_id in BENCHMARK_STYLE_IDS
}

# Cache Fs once per style reference.
image_size = {'0.06M': 256, '0.25M': 512, '1M': 1024}.get(config.model_pn, 512)
style_trace_cache = {}
for style_id, style_case in tqdm(style_cases.items(), desc='VQVAE style traces'):
    style_trace_cache[style_id] = [
        _cpu(feature) for feature in encode_style_reference(
            bundle, style_case.style_reference_path, image_size
        )
    ]

# Cache baseline image and Fc trajectory once per content prompt.
content_cache = {}
for content_id, content_case in tqdm(content_cases.items(), desc='Clean content traces'):
    baseline = _cpu(_run_infinity(
        bundle.infinity_model.autoregressive_infer_cfg,
        **_infer_kwargs(bundle, config, [content_case.content_prompt], config.seed),
    ))
    content_dir = BENCHMARK_DIR / content_id
    content_dir.mkdir(parents=True, exist_ok=True)
    baseline_path = content_dir / 'baseline.png'
    _image(baseline, 0).save(baseline_path)
    content_cache[content_id] = {
        'trace': baseline[3], 'baseline_path': baseline_path,
        'content_prompt': content_case.content_prompt,
    }

generation_rows = []
total_runs = len(BENCHMARK_CONTENT_IDS) * len(BENCHMARK_STYLE_IDS) * len(BENCHMARK_STEPS) * 2
progress = tqdm(total=total_runs, desc='PFB / Content Projection generations')
for content_id in BENCHMARK_CONTENT_IDS:
    cache = content_cache[content_id]
    fc_trace = cache['trace']
    prompt = cache['content_prompt']
    paired_kwargs = _infer_kwargs(bundle, config, [prompt, prompt], config.seed)
    for style_id in BENCHMARK_STYLE_IDS:
        fs_trace = style_trace_cache[style_id]
        style_case = style_cases[style_id]
        evaluation_prompt = f'{prompt}, in {style_case.style_label} style'
        for step in BENCHMARK_STEPS:
            step_dir = BENCHMARK_DIR / content_id / style_id / f'step_{step:02d}'
            step_dir.mkdir(parents=True, exist_ok=True)
            baseline_copy = step_dir / 'baseline.png'
            if not baseline_copy.exists():
                shutil.copy2(cache['baseline_path'], baseline_copy)

            pfb_path = step_dir / 'pfb.png'
            if not pfb_path.exists():
                pfb_feature = principal_feature_blend(
                    fc_trace[step], fs_trace[step], rank=1,
                    alpha=config.paper_alpha, strength=1.0,
                )
                pfb_result = _cpu(_run_infinity(
                    bundle.infinity_model.autoregressive_infer_pfb,
                    **paired_kwargs, pfb_feature=pfb_feature, inject_step=step,
                    f_con=fc_trace, sac=False,
                ))
                _image(pfb_result, 1).save(pfb_path)
                del pfb_result, pfb_feature
            progress.update(1)

            projection_diagnostics = []
            projection_path = step_dir / 'content_projection.png'
            if not projection_path.exists():
                projection_result = _cpu(_run_infinity(
                    bundle.infinity_model.autoregressive_infer_content_projection,
                    **paired_kwargs, style_feature=fs_trace[step], inject_step=step,
                    f_con=fc_trace, style_rank=1, content_rank=1,
                    alpha=config.paper_alpha, strength=1.0, projection_strength=1.0,
                    preserve_mean=False, projection_diagnostics=projection_diagnostics, sac=False,
                ))
                _image(projection_result, 1).save(projection_path)
                del projection_result
            progress.update(1)

            common = {
                'content_id': content_id, 'style_id': style_id, 'step': step,
                'content_prompt': prompt, 'evaluation_prompt': evaluation_prompt,
                'style_reference_path': str(style_case.style_reference_path),
            }
            generation_rows.extend([
                {**common, 'method': 'Baseline', 'image_path': str(baseline_copy)},
                {**common, 'method': 'PFB', 'image_path': str(pfb_path)},
                {**common, 'method': 'Content Projection', 'image_path': str(projection_path),
                 'projection_diagnostics': json.dumps(projection_diagnostics)},
            ])
progress.close()
gc.collect()
torch.cuda.empty_cache()
print(f'Saved {len(generation_rows)} evaluation rows under {BENCHMARK_DIR}')

## 11. CLIP prompt fidelity, style-image fidelity, and harmonic score

`S_txt` compares each generated image with the content-plus-style evaluation prompt. `S_img` compares it with the corresponding CSD100 style-reference image. The harmonic mean penalizes methods that improve only one side.

In [ ]:
import pandas as pd
from var_soict.clip_metrics import CLIPMetricsEvaluator

METRICS_DIR = BENCHMARK_DIR / 'metrics'
evaluator = CLIPMetricsEvaluator(output_dir=METRICS_DIR)
metric_rows = []
for row in tqdm(generation_rows, desc='CLIP metrics'):
    generated = evaluator.image_embedding(row['image_path'])
    prompt_embedding = evaluator.text_embedding(row['evaluation_prompt'])
    style_embedding = evaluator.image_embedding(row['style_reference_path'])
    s_txt = evaluator.cosine_score(generated, prompt_embedding)
    s_img = evaluator.cosine_score(generated, style_embedding)
    metric_rows.append({
        **row, 'S_txt': s_txt, 'S_img': s_img,
        'S_harmonic': evaluator.harmonic_score(s_txt, s_img),
    })

metrics_df = pd.DataFrame(metric_rows)
summary_df = (
    metrics_df.groupby(['method', 'step'], as_index=False)
    .agg(
        num_pairs=('image_path', 'size'),
        S_txt=('S_txt', 'mean'),
        S_img=('S_img', 'mean'),
        S_harmonic=('S_harmonic', 'mean'),
    )
    .sort_values(['step', 'method'])
)
by_pair_df = (
    metrics_df.groupby(['content_id', 'style_id', 'method', 'step'], as_index=False)
    [['S_txt', 'S_img', 'S_harmonic']].mean()
)
metrics_df.to_csv(METRICS_DIR / 'clip_metrics_detail.csv', index=False)
summary_df.to_csv(METRICS_DIR / 'clip_metrics_by_method_step.csv', index=False)
by_pair_df.to_csv(METRICS_DIR / 'clip_metrics_by_pair.csv', index=False)

display(summary_df.style.format({
    'S_txt': '{:.4f}', 'S_img': '{:.4f}', 'S_harmonic': '{:.4f}',
}))
display(summary_df.pivot(index='step', columns='method', values='S_harmonic').style.format('{:.4f}'))

projection_summary = summary_df[summary_df['method'] == 'Content Projection']
best_projection_row = projection_summary.loc[projection_summary['S_harmonic'].idxmax()]
print(
    'Best Content Projection step:', int(best_projection_row['step']),
    '| harmonic:', f"{best_projection_row['S_harmonic']:.4f}",
)
print('Detailed metrics:', METRICS_DIR / 'clip_metrics_detail.csv')
print('Summary:', METRICS_DIR / 'clip_metrics_by_method_step.csv')